In [1]:
import pyomo.environ as pyo

import pandas as pd
import numpy as np


In [2]:
SOLVER = pyo.SolverFactory('appsi_highs')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)


Solver ready: <pyomo.contrib.appsi.base.SolverFactoryClass.register.<locals>.decorator.<locals>.LegacySolver object at 0x000001C729011010>


# Question 1
Part a: Deterministic Model for Average Seat Demand

$d_cs$ be the demand for seat class c during scenario s 
$s_cs$ be the number of seats to allocate for seat class c during scenario s
$t_cs$ be the number of tickets sold for seat class c duriong scenario s 

$c = \{0,1,2\}$, where 0 is economy, 1 is first, 2 is buisness class. 
$s = \{0,1,2\}$, where 0 is weekday morning/evening, 1 is weekend, 2 is weekday midday

\begin{align*}
\min \quad &1/3 \sum_{s \in S} t_{0s} + 3 t_{1s} + 2 t_{2s}\\ 
\text{s.t} \quad &s_0 + 2s_1 + 1.5s_2 \le 200 \\
&t_{cs} \le d_{cs} \forall c,s \\
&t_{cs} \le s_{cs} \forall cs 
\end{align*}

Part b: Two Step Stochastic Model

In [ ]:

model = pyo.ConcreteModel()

# Sets
model.C = pyo.Set(initialize=['economy', 'business', 'first'])
model.S = pyo.Set(initialize=[1,2,3])  # weekday morning/evening, weekend, weekday midday

# Data
profit = {'economy': 1, 'business': 2, 'first': 3}
weight = {'economy': 1, 'business': 1.5, 'first': 2}
prob = 1/3

# Demand[class, scenario]
demand = {
    ('first', 1): 20, ('business', 1): 50, ('economy', 1): 200,
    ('first', 2): 10, ('business', 2): 24, ('economy', 2): 175,
    ('first', 3): 6,  ('business', 3): 10, ('economy', 3): 150
}

# Variables
model.seats = pyo.Var(model.C, domain=pyo.NonNegativeIntegers) # Seats
model.tickets = pyo.Var(model.C, model.S, domain=pyo.NonNegativeReals) # Tickets sold

# Objective: Maximize expected profit
model.obj = pyo.Objective(
    expr = sum(prob * sum(profit[c] * model.tickets[c, s] for c in model.C) for s in model.S),
    sense = pyo.maximize
)

# Constraints
model.capacity = pyo.Constraint(
    expr = sum(weight[c] * model.seats[c] for c in model.C) <= 200
)

def seat_limit_rule(m, c, s):
    return m.tickets[c, s] <= m.seats[c]
model.seat_limit = pyo.Constraint(model.C, model.S, rule=seat_limit_rule)

def demand_limit_rule(m, c, s):
    return m.tickets[c, s] <= demand[c, s]
model.demand_limit = pyo.Constraint(model.C, model.S, rule=demand_limit_rule)

results = SOLVER.solve(model)

for c in model.C:
    print(f"{c} class: {model.seats[c].value:.0f} seats allocated")
    
print(f"Expected profit: {pyo.value(model.obj):.2f}")

economy class: 150 seats allocated
business class: 20 seats allocated
first class: 10 seats allocated
Expected profit: 209.33


# Problem 2

Q1: Write down 2 stage optimization model for Expected Total Cost

\begin{align*}
\min \quad &\sum_{i \in I}c_i g_i + c^wE[w_s] \\
\text{s.t} \quad & g_i \ge g_i^{min} \\
& g_i \le g_i^{max}\\ 
& w_s \le w^{max}_s \\
& \sum_{i \in I} g_i + w_s \ge d_s \\
& w \ge 0
\end{align*}

In [95]:
model = pyo.ConcreteModel()

c1 = 50
c2 = 100
cw = 40
gen_min = {1: 0, 2: 300}
gen_max = {1: 1000, 2: 1000}


model.I = pyo.Set(initialize=[1,2]) # Generators
model.costs = pyo.Param(model.I, initialize={1: c1, 2: c2})
model.gen_min = pyo.Param(model.I, initialize=gen_min)
model.gen_max = pyo.Param(model.I, initialize=gen_max)

model.gen_output = pyo.Var(model.I, domain=pyo.NonNegativeReals)

model.S = pyo.Set(initialize=range(100)) # Scenarios
model.wind = pyo.Var(model.S, domain=pyo.NonNegativeReals)
model.demand = pyo.Param(model.S,
                         initialize=[np.random.uniform(1300, 1600) for _ in model.S])

model.wind_max = pyo.Param(model.S,
                         initialize=[np.random.uniform(400, 500) for _ in model.S])

#generator constraints
def min_gen_output(m, i):
    return m.gen_output[i] >= m.gen_min[i]
model.min_generator_output = pyo.Constraint(model.I, rule=min_gen_output)

def max_gen_output(m, i):
    return m.gen_output[i] <= m.gen_max[i]
model.max_generator_ouput = pyo.Constraint(model.I, rule=max_gen_output)

def wind_limit(m, s):
    return m.wind[s] <= m.wind_max[s]
model.wind_limit = pyo.Constraint(model.S, rule=wind_limit)

def demand_constraint(m, s):
    return sum(m.gen_output[i] for i in m.I) + m.wind[s] >= m.demand[s]
model.demand_constraint = pyo.Constraint(model.S, rule=demand_constraint)

model.obj = pyo.Objective(
    expr = sum(model.costs[i] * model.gen_output[i] for i in model.I) + 1/100 * sum(cw * model.wind[s] for s in model.S),
    sense = pyo.minimize
)
    
    
result = SOLVER.solve(model)

In [99]:
print(f"Optimal Cost: {model.obj()}")
print(f"Generator 1 output: {model.gen_output[1].value:.2f}")
print(f"Generator 2 output: {model.gen_output[2].value:.2f}")

pd.DataFrame({
    'Scenario': list(model.S),
    'Wind Output': [model.wind[s].value for s in model.S],
    'Max Wind': [model.wind_max[s] for s in model.S],
    'Demand': [model.demand[s] for s in model.S]
})

Optimal Cost: 85110.31349418704
Generator 1 output: 891.78
Generator 2 output: 300.00


,Scenario,Wind Output,Max Wind,Demand
0,0,139.160799,469.816171,1330.937161
1,1,378.989510,453.609637,1570.765872
2,2,259.799350,430.952762,1451.575712
3,3,356.160878,481.379502,1547.937240
4,4,204.238519,468.473117,1396.014880
...,...,...,...,...
95,95,214.229307,447.396164,1406.005668
96,96,283.320472,466.755774,1475.096834
97,97,131.544029,417.231987,1323.320391
98,98,400.542081,419.228902,1592.318442
